# Financial Market Data Collection & Preprocessing

## Objective

The objective of this notebook is to build a clean and structured financial dataset for future quantitative finance research tasks including:

- Cointegration analysis
- Engle-Granger testing
- Johansen testing
- Spread analysis
- Rolling window stability analysis
- Statistical arbitrage research

The dataset consists of major US equities from the S&P 500 along with sector and market ETFs downloaded using Yahoo Finance.

## Import Libraries 

In [56]:
import yfinance as yf
import pandas as pd
import numpy as np

import requests
from bs4 import BeautifulSoup
from io import StringIO

In [57]:
# stocks = [

#     # Technology
#     'AAPL','MSFT','NVDA','GOOG','META','AMZN','TSLA',
#     'AMD','INTC','CSCO','ORCL','CRM','ADBE','QCOM',
#     'TXN','AVGO','IBM','AMAT','MU','NOW',

#     # Banking & Finance
#     'JPM','BAC','WFC','GS','MS','C','AXP','BLK',
#     'SCHW','USB','PNC','TFC',

#     # Energy
#     'XOM','CVX','COP','SLB','EOG','PSX','MPC',

#     # Healthcare
#     'JNJ','PFE','MRK','ABBV','TMO','LLY','ABT',
#     'DHR','BMY','UNH','CVS',

#     # Consumer
#     'KO','PEP','PG','COST','WMT','HD','MCD',
#     'SBUX','NKE','LOW','TGT',

#     # Industrials
#     'CAT','BA','HON','GE','MMM','UPS','FDX',
#     'LMT','DE',

#     # Telecom & Media
#     'VZ','T','CMCSA','DIS','NFLX',

#     # Utilities
#     'NEE','DUK','SO',

#     # Real Estate
#     'AMT','PLD','SPG',

#     # Materials
#     'LIN','APD','FCX',

#     # Transportation
#     'UNP','CSX',

#     # Semiconductor
#     'NXPI','KLAC','LRCX',

#     # Additional large caps
#     'PYPL','UBER','SHOP','ZM'
# ]

## Retrieve top 100 S&P 500 Companies 
The list of companies was extracted from the S&P 500 Wikipedia dataset instead of manually specifying individual stock names. This approach ensures a more systematic, scalable, and reproducible selection of top US market companies.


In [58]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, 'html.parser')

table = soup.find('table', {'id': 'constituents'})

sp500 = pd.read_html(
    StringIO(str(table))
)[0]

sp500.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [59]:
import yfinance as yf
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

symbols = [s.replace('.', '-') for s in sp500['Symbol']]

def get_market_cap(symbol):
    try:
        ticker = yf.Ticker(symbol)
        # Fast history call or fast info if available, but standard info works better in parallel
        return {"Symbol": symbol, "MarketCap": ticker.info.get("marketCap", None)}
    except Exception:
        return {"Symbol": symbol, "MarketCap": None}

market_caps = []

# Use ThreadPoolExecutor to make multiple network requests at the same time
# max_workers=20 is a safe sweet spot to avoid getting blocked by Yahoo
with ThreadPoolExecutor(max_workers=20) as executor:
    futures = [executor.submit(get_market_cap, symbol) for symbol in symbols]
    
    for future in as_completed(futures):
        market_caps.append(future.result())


market_cap_df = pd.DataFrame(market_caps)

In [60]:
market_cap_df.head()

market_cap_df.shape

(503, 2)

In [63]:
market_cap_df = market_cap_df.dropna()

top100 = (
    market_cap_df
    .sort_values("MarketCap", ascending=False)
    .head(100)
)

top100.head(20)

,Symbol,MarketCap
350,NVDA,5201459675136
36,AAPL,4556898959360
14,GOOGL,4349323116544
24,GOOG,4309220851712
315,MSFT,3174467371008
21,AMZN,2689492582400
74,AVGO,2268995256320
441,TSLA,1591300194304
311,META,1581386956800
314,MU,1217467777024


In [64]:
top100.to_csv(
    "../data/raw/top100_us_stocks.csv",
    index=False
)

In [65]:
stocks = top100["Symbol"].tolist()

print(len(stocks))

100


## Selection and Formatting of Stock Symbols

The stock symbols of the first 100 S&P 500 companies are extracted from the dataset to create the research universe for analysis. Certain ticker symbols containing periods (.) are converted to hyphen (-) format to ensure compatibility with Yahoo Finance data retrieval conventions.


In [66]:
stocks = top100["Symbol"].tolist()



stocks = [stock.replace('.', '-') for stock in stocks]

stocks[:30]

['NVDA',
 'AAPL',
 'GOOGL',
 'GOOG',
 'MSFT',
 'AMZN',
 'AVGO',
 'TSLA',
 'META',
 'MU',
 'BRK-B',
 'LLY',
 'WMT',
 'AMD',
 'JPM',
 'ORCL',
 'XOM',
 'V',
 'INTC',
 'JNJ',
 'CSCO',
 'LRCX',
 'CAT',
 'COST',
 'MA',
 'AMAT',
 'ABBV',
 'CVX',
 'BAC',
 'NFLX']

In [67]:
print("Total symbols:", len(top100["Symbol"]))
print("Unique symbols:", top100["Symbol"].nunique())

Total symbols: 100
Unique symbols: 100


## Add Major ETFs

Major ETFs are manually included to provide representation of broad market indices and sector-specific benchmarks that are not part of the S&P 500 company list. These ETFs help capture overall market behavior, sector movements, and diversified market exposure for future correlation and cointegration analysis.


In [68]:
etfs = [
    'SPY','QQQ','DIA','IWM',
    'XLF','XLK','XLE','XLV',
    'XLY','XLP','XLI','XLU',
    'VNQ','GLD','TLT'
]

In [69]:
symbols = stocks + etfs

len(symbols)

115

## Historical Market Data Collection

Historical price data is downloaded using the Yahoo Finance API through the yfinance library. The dataset includes daily market data from 2018 onward, while the end date is dynamically generated.
The dataset begins in 2018 to ensure sufficient historical coverage across multiple market regimes, including:

- Bull market periods
- COVID-19 market crash
- Recovery phases
- Inflationary environments


In [70]:
print("AXP" in symbols)
print("BRK-B" in symbols)

True
True


In [71]:
start_date = '2018-01-01'

# Dynamic latest date
end_date = pd.Timestamp.today().strftime('%Y-%m-%d')

data = yf.download(
    symbols,
    start=start_date,
    end=end_date,
    auto_adjust=False
)

[*********************100%***********************]  115 of 115 completed


In [72]:
import yfinance as yf
end_date = pd.Timestamp.today().strftime('%Y-%m-%d')

axp = yf.download(
    "AXP",
    start="2018-01-01",
    end=end_date,
    auto_adjust=False
)

print(axp.shape)
axp.head()

[*********************100%***********************]  1 of 1 completed

(2116, 6)


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,AXP,AXP,AXP,AXP,AXP,AXP
2018-01-02,88.292404,98.940002,99.730003,98.220001,99.730003,2746700
2018-01-03,88.836761,99.550003,99.760002,99.019997,99.239998,2976400
2018-01-04,90.314415,100.849998,101.650002,99.910004,100.029999,3122000
2018-01-05,90.520393,101.080002,101.080002,100.269997,100.849998,2550300
2018-01-08,89.687538,100.150002,101.199997,100.120003,101.000000,1896500


In [73]:
brk = yf.download(
    "BRK-B",
    start="2018-01-01",
    end=end_date,
    auto_adjust=False
)

print(brk.shape)
brk.head()

[*********************100%***********************]  1 of 1 completed

(2116, 6)


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,BRK-B,BRK-B,BRK-B,BRK-B,BRK-B,BRK-B
2018-01-02,197.220001,197.220001,198.869995,195.960007,198.869995,4113000
2018-01-03,199.789993,199.789993,200.000000,197.000000,197.000000,3526700
2018-01-04,200.690002,200.690002,202.130005,200.009995,200.899994,3900000
2018-01-05,201.419998,201.419998,201.669998,199.309998,201.000000,4207600
2018-01-08,202.740005,202.740005,203.070007,199.800003,201.149994,3887900


In [75]:
axp.shape
brk.shape

(2116, 6)

In [76]:
print(axp.columns)
print()
print(brk.columns)

MultiIndex([('Adj Close', 'AXP'),
            (    'Close', 'AXP'),
            (     'High', 'AXP'),
            (      'Low', 'AXP'),
            (     'Open', 'AXP'),
            (   'Volume', 'AXP')],
           names=['Price', 'Ticker'])

MultiIndex([('Adj Close', 'BRK-B'),
            (    'Close', 'BRK-B'),
            (     'High', 'BRK-B'),
            (      'Low', 'BRK-B'),
            (     'Open', 'BRK-B'),
            (   'Volume', 'BRK-B')],
           names=['Price', 'Ticker'])


In [77]:
print(type(axp.columns))
print(type(data.columns))

<class 'pandas.MultiIndex'>
<class 'pandas.MultiIndex'>


In [78]:
axp.head()

Price,Adj Close,Close,High,Low,Open,Volume
Ticker,AXP,AXP,AXP,AXP,AXP,AXP
2018-01-02,88.292404,98.940002,99.730003,98.220001,99.730003,2746700
2018-01-03,88.836761,99.550003,99.760002,99.019997,99.239998,2976400
2018-01-04,90.314415,100.849998,101.650002,99.910004,100.029999,3122000
2018-01-05,90.520393,101.080002,101.080002,100.269997,100.849998,2550300
2018-01-08,89.687538,100.150002,101.199997,100.120003,101.000000,1896500


In [79]:
print(axp.columns)
print(brk.columns)

MultiIndex([('Adj Close', 'AXP'),
            (    'Close', 'AXP'),
            (     'High', 'AXP'),
            (      'Low', 'AXP'),
            (     'Open', 'AXP'),
            (   'Volume', 'AXP')],
           names=['Price', 'Ticker'])
MultiIndex([('Adj Close', 'BRK-B'),
            (    'Close', 'BRK-B'),
            (     'High', 'BRK-B'),
            (      'Low', 'BRK-B'),
            (     'Open', 'BRK-B'),
            (   'Volume', 'BRK-B')],
           names=['Price', 'Ticker'])


In [80]:
axp[('Adj Close', 'AXP')].head()

2018-01-02    88.292404
2018-01-03    88.836761
2018-01-04    90.314415
2018-01-05    90.520393
2018-01-08    89.687538
Name: (Adj Close, AXP), dtype: float64

In [81]:
brk[('Adj Close', 'BRK-B')].head()

2018-01-02    197.220001
2018-01-03    199.789993
2018-01-04    200.690002
2018-01-05    201.419998
2018-01-08    202.740005
Name: (Adj Close, BRK-B), dtype: float64

In [82]:
adj_close = data['Adj Close'].copy()

In [83]:
adj_close.loc[:, 'AXP'] = axp[('Adj Close', 'AXP')]

adj_close.loc[:, 'BRK-B'] = brk[('Adj Close', 'BRK-B')]

In [84]:
adj_close[['AXP', 'BRK-B']].isnull().sum()

Ticker
AXP      0
BRK-B    0
dtype: int64

In [85]:
adj_close[['AXP', 'BRK-B']].head()

Ticker,AXP,BRK-B
2018-01-02,88.292404,197.220001
2018-01-03,88.836761,199.789993
2018-01-04,90.314415,200.690002
2018-01-05,90.520393,201.419998
2018-01-08,89.687538,202.740005


In [86]:
adj_close[['MSFT','NVDA','META','AXP','BRK-B']].isnull().sum()

Ticker
MSFT     0
NVDA     0
META     0
AXP      0
BRK-B    0
dtype: int64

## Extraction of Adjusted Closing Prices

Adjusted closing prices are extracted from the dataset as they account for stock splits and dividends, making them more reliable for accurate financial and time series analysis.


In [87]:
adj_close.head()

Ticker,AAPL,ABBV,ABT,ADI,AMAT,AMD,AMGN,AMZN,ANET,APH,...,WMT,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,XOM
2018-01-02,40.267067,68.773155,50.506084,77.390320,48.314182,10.98,136.742554,59.450500,14.439375,20.302286,...,28.800240,25.747269,23.884144,66.254860,29.872919,45.314571,20.132584,72.723335,46.275230,58.185490
2018-01-03,40.260067,69.849380,50.617760,78.350388,49.170441,11.55,139.322861,60.209999,14.725000,20.565889,...,29.051455,26.132860,24.012457,66.611687,30.122101,45.298546,19.974421,73.419182,46.487701,59.328255
2018-01-04,40.447071,69.451027,50.531860,78.264664,49.452831,12.12,138.735733,60.479500,14.543125,20.494209,...,29.077751,26.290604,24.234869,67.099106,30.274374,45.426777,19.808550,73.523544,46.640125,59.410374
2018-01-05,40.907574,70.660011,50.677898,78.581848,49.735207,11.88,139.562332,61.457001,14.798125,20.743937,...,29.250095,26.280088,24.303308,67.560448,30.592762,45.627121,19.800835,74.149803,47.009621,59.362499
2018-01-08,40.755627,69.527901,50.531860,78.719002,50.937595,12.28,139.523773,62.343498,15.691250,20.965916,...,29.682436,26.437826,24.269100,67.838974,30.708107,45.739334,19.985996,73.880165,47.065033,59.629337


## Verification of Downloaded Symbols

The downloaded symbols are verified against the expected symbol list to identify missing or failed ticker downloads and ensure dataset completeness.


In [88]:
print("Expected Symbols:", len(symbols))

print("Downloaded Symbols:", len(adj_close.columns))

missing_symbols = set(symbols) - set(adj_close.columns)

if missing_symbols:
    print("Warning: Missing symbols detected")
    print(missing_symbols)
else:
    print("All symbols downloaded successfully")

Expected Symbols: 115
Downloaded Symbols: 115
All symbols downloaded successfully


## Save Raw Market Dataset

The complete raw market dataset is saved to preserve all downloaded fields for future research tasks including volatility analysis, liquidity analysis, and spread studies.

In [89]:
print(adj_close.shape)

adj_close[['MSFT','NVDA','META','AXP','BRK-B']].isnull().sum()

(2116, 115)


Ticker
MSFT     0
NVDA     0
META     0
AXP      0
BRK-B    0
dtype: int64

In [90]:
data.to_csv(
    '../data/raw/full_market_data.csv'
)

## Verify Datetime Index

The index is converted to datetime format to support rolling-window analysis, plotting, and statistical testing.

In [91]:
adj_close.index = pd.to_datetime(adj_close.index)

adj_close.index

DatetimeIndex(['2018-01-02', '2018-01-03', '2018-01-04', '2018-01-05',
               '2018-01-08', '2018-01-09', '2018-01-10', '2018-01-11',
               '2018-01-12', '2018-01-16',
               ...
               '2026-05-20', '2026-05-21', '2026-05-22', '2026-05-26',
               '2026-05-27', '2026-05-28', '2026-05-29', '2026-06-01',
               '2026-06-02', '2026-06-03'],
              dtype='datetime64[s]', length=2116, freq=None)

## Check for Duplicate Dates

Duplicate dates are identified and removed to maintain dataset consistency and ensure accurate time series analysis.


In [92]:
print(
    "Duplicate Dates:",
    adj_close.index.duplicated().sum()
)

adj_close = adj_close[
    ~adj_close.index.duplicated()
]

Duplicate Dates: 0


## Check for any missing value


In [93]:
adj_close.isnull().sum()

Ticker
AAPL    0
ABBV    0
ABT     0
ADI     0
AMAT    0
       ..
XLP     0
XLU     0
XLV     0
XLY     0
XOM     0
Length: 115, dtype: int64

## Missing value Percentage


In [94]:
missing_pct = (
    adj_close.isnull().mean() * 100
)

missing_pct.sort_values(
    ascending=False
)

Ticker
SNDK    84.546314
GEV     74.102079
APP     39.035917
PLTR    32.655955
CRWD    17.107750
          ...    
GE       0.000000
ETN      0.000000
DIS      0.000000
DIA      0.000000
XOM      0.000000
Length: 115, dtype: float64

In [95]:
removed_assets = missing_pct[missing_pct >= 5]

print( "Removed Assets:")

print(removed_assets.sort_values( ascending=False))

Removed Assets:
Ticker
SNDK    84.546314
GEV     74.102079
APP     39.035917
PLTR    32.655955
CRWD    17.107750
UBER    16.068053
VRT      6.947070
dtype: float64


In [96]:
removed_assets.to_csv(
    "../data/processed/removed_assets.csv"
)

## Filtering Assets with Excessive Missing Values

Assets containing more than 5% missing observations are removed to maintain dataset consistency and improve the reliability of future financial analysis.
A 5% threshold is used to balance data completeness.

In [97]:
missing_pct.sort_values(
    ascending=False
).head(20)

Ticker
SNDK    84.546314
GEV     74.102079
APP     39.035917
PLTR    32.655955
CRWD    17.107750
UBER    16.068053
VRT      6.947070
SCHW     0.000000
RTX      0.000000
QQQ      0.000000
QCOM     0.000000
PM       0.000000
PLD      0.000000
PG       0.000000
PFE      0.000000
PEP      0.000000
AAPL     0.000000
SPGI     0.000000
ORCL     0.000000
NVDA     0.000000
dtype: float64

In [98]:
valid_cols = missing_pct[
    missing_pct < 5
].index

adj_close = adj_close[valid_cols]

## Handling Remaining Missing Values

Remaining missing values are filled using forward-fill and backward-fill methods to create a complete dataset for time series analysis.


In [99]:
adj_close = adj_close.ffill()

In [100]:
# Final check for any remaining missing values
adj_close.isnull().sum().sum()

0

## Validation check 
check fir any zero or negative price values

In [101]:
(adj_close <= 0).sum().sum()

0

In [102]:
# download the cleaned adjusted close data
adj_close.to_csv(
    '../data/cleaned/adj_close_cleaned.csv'
)

In [103]:
# Calculate daily returns
returns = adj_close.pct_change().dropna()

returns.head()

Ticker,AAPL,ABBV,ABT,ADI,AMAT,AMD,AMGN,AMZN,ANET,APH,...,WMT,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,XOM
2018-01-03,-0.000174,0.015649,0.002211,0.012406,0.017723,0.051913,0.018870,0.012775,0.019781,0.012984,...,0.008723,0.014976,0.005372,0.005386,0.008341,-0.000354,-0.007856,0.009568,0.004591,0.019640
2018-01-04,0.004645,-0.005703,-0.001697,-0.001094,0.005743,0.049351,-0.004214,0.004476,-0.012351,-0.003485,...,0.000905,0.006036,0.009262,0.007317,0.005055,0.002831,-0.008304,0.001421,0.003279,0.001384
2018-01-05,0.011385,0.017408,0.002890,0.004053,0.005710,-0.019802,0.005958,0.016163,0.017534,0.012185,...,0.005927,-0.000400,0.002824,0.006876,0.010517,0.004410,-0.000389,0.008518,0.007922,-0.000806
2018-01-08,-0.003714,-0.016022,-0.002882,0.001745,0.024176,0.033670,-0.000276,0.014425,0.060354,0.010701,...,0.014781,0.006002,-0.001408,0.004123,0.003770,0.002459,0.009351,-0.003636,0.001179,0.004495
2018-01-09,-0.000115,0.007539,0.001700,-0.002069,-0.018956,-0.037459,0.015393,0.004676,-0.004302,0.001544,...,-0.012006,-0.002519,0.007754,0.006415,-0.002554,-0.001402,-0.009844,0.011773,0.001963,-0.004246


## Generation of Log Returns

Log returns are calculated to support quantitative financial analysis and time series modeling due to their favorable mathematical and statistical properties.


In [104]:
log_returns = np.log(
    adj_close / adj_close.shift(1)
).dropna()

## Price Normalization

Asset prices are normalized to a common starting value for meaningful comparison of price movements across different stocks during exploratory data analysis.


In [105]:
normalized_prices = (
    adj_close / adj_close.iloc[0]
)

normalized_prices.head()

Ticker,AAPL,ABBV,ABT,ADI,AMAT,AMD,AMGN,AMZN,ANET,APH,...,WMT,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,XOM
2018-01-02,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
2018-01-03,0.999826,1.015649,1.002211,1.012406,1.017723,1.051913,1.018870,1.012775,1.019781,1.012984,...,1.008723,1.014976,1.005372,1.005386,1.008341,0.999646,0.992144,1.009568,1.004591,1.019640
2018-01-04,1.004470,1.009857,1.000510,1.011298,1.023568,1.103825,1.014576,1.017309,1.007185,1.009453,...,1.009636,1.021103,1.014684,1.012742,1.013439,1.002476,0.983905,1.011003,1.007885,1.021051
2018-01-05,1.015906,1.027436,1.003402,1.015396,1.029412,1.081967,1.020621,1.033751,1.024845,1.021754,...,1.015620,1.020694,1.017550,1.019706,1.024097,1.006897,0.983522,1.019615,1.015870,1.020229
2018-01-08,1.012133,1.010974,1.000510,1.017169,1.054299,1.118397,1.020339,1.048662,1.086699,1.032687,...,1.030632,1.026821,1.016118,1.023909,1.027958,1.009374,0.992719,1.015907,1.017068,1.024815


In [106]:
normalized_prices.to_csv(
    '../data/processed/normalized_prices.csv'
)

## Rolling Volatility Estimation

Rolling 30-day volatility is calculated to analyze changes in asset risk and price variability over time for future financial and stability analysis. A 30-day window is used because it approximately represents one trading month, making it suitable for capturing short-term market risk dynamics.


In [107]:
rolling_volatility = (
    returns.rolling(30).std()
)

rolling_volatility.head()

Ticker,AAPL,ABBV,ABT,ADI,AMAT,AMD,AMGN,AMZN,ANET,APH,...,WMT,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,XOM
2018-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [108]:
rolling_volatility.to_csv(
    '../data/processed/rolling_volatility.csv'
)

## Save Sector Metadata
Sector information is extracted and saved to support sector-wise analysis, asset grouping, and future cointegration and relationship studies between economically related companies.

In [109]:
sector_info = sp500[
    ['Symbol', 'GICS Sector']
]

sector_info['Symbol'] = (
    sector_info['Symbol']
    .str.replace('.', '-', regex=False)
)

sector_info.to_csv(
    '../data/processed/sector_info.csv',
    index=False
)

sector_info.head()

,Symbol,GICS Sector
0,MMM,Industrials
1,AOS,Industrials
2,ABT,Health Care
3,ABBV,Health Care
4,ACN,Information Technology


In [110]:
returns.to_csv(
    '../data/processed/returns.csv'
)

log_returns.to_csv(
    '../data/processed/log_returns.csv'
)

In [111]:
returns.describe()

Ticker,AAPL,ABBV,ABT,ADI,AMAT,AMD,AMGN,AMZN,ANET,APH,...,WMT,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,XOM
count,2115.000000,2115.000000,2115.000000,2115.000000,2115.000000,2115.000000,2115.000000,2115.000000,2115.000000,2115.000000,...,2115.000000,2115.000000,2115.000000,2115.000000,2115.000000,2115.000000,2115.000000,2115.000000,2115.000000,2115.000000
mean,0.001150,0.000687,0.000378,0.001053,0.001510,0.002463,0.000558,0.000912,0.001606,0.001115,...,0.000763,0.000588,0.000466,0.000547,0.001027,0.000330,0.000448,0.000395,0.000549,0.000636
std,0.019207,0.016810,0.015548,0.021658,0.028356,0.035340,0.016122,0.021559,0.029101,0.018713,...,0.014193,0.019815,0.014676,0.013394,0.016525,0.009807,0.012762,0.010972,0.014896,0.018997
min,-0.128647,-0.162524,-0.100389,-0.166149,-0.203576,-0.173144,-0.082607,-0.140494,-0.242344,-0.138752,...,-0.113758,-0.201412,-0.137093,-0.113441,-0.138140,-0.093956,-0.113577,-0.098610,-0.126686,-0.122248
25%,-0.007858,-0.007053,-0.007069,-0.010262,-0.013297,-0.016697,-0.007283,-0.010377,-0.012355,-0.007663,...,-0.005996,-0.008881,-0.005891,-0.005524,-0.007084,-0.004285,-0.005873,-0.004979,-0.006429,-0.009471
50%,0.001175,0.001127,0.000669,0.001195,0.001389,0.000683,0.000352,0.001163,0.001991,0.001519,...,0.000855,0.001031,0.000834,0.000949,0.001741,0.000554,0.000966,0.000568,0.001324,0.000583
75%,0.010921,0.009183,0.008565,0.012637,0.017172,0.020647,0.008498,0.012306,0.016443,0.010383,...,0.007559,0.010268,0.007486,0.007316,0.009845,0.005348,0.006960,0.006031,0.008437,0.010767
max,0.153288,0.137673,0.109360,0.183876,0.161058,0.238205,0.118180,0.135359,0.203882,0.101715,...,0.117085,0.160373,0.131566,0.126512,0.134257,0.085106,0.127934,0.077057,0.108880,0.126868


In [112]:
returns.max()

returns.min()

Ticker
AAPL   -0.128647
ABBV   -0.162524
ABT    -0.100389
ADI    -0.166149
AMAT   -0.203576
          ...   
XLP    -0.093956
XLU    -0.113577
XLV    -0.098610
XLY    -0.126686
XOM    -0.122248
Length: 108, dtype: float64

## Dataset summary

In [113]:
print("FINAL DATASET SUMMARY")

print("-" * 40)

print(
    "Number of Assets:",
    adj_close.shape[1]
)

print(
    "Trading Days:",
    adj_close.shape[0]
)

print(
    "Start Date:",
    adj_close.index.min()
)

print(
    "End Date:",
    adj_close.index.max()
)

print(
    "Remaining Missing Values:",
    adj_close.isnull().sum().sum()
)

FINAL DATASET SUMMARY
----------------------------------------
Number of Assets: 108
Trading Days: 2116
Start Date: 2018-01-02 00:00:00
End Date: 2026-06-03 00:00:00
Remaining Missing Values: 0
